<a href="https://colab.research.google.com/github/manishsharma3574/stock-price-prediction-using-machine-learning/blob/main/cleaning%20data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np

# =========================
# LOAD DATA
df = pd.read_excel("/faang_stock_prices (1).csv.xlsx")   # Fix path if needed

# =========================
# DATA CLEANING
df['Date'] = pd.to_datetime(df['Date'])

# Remove invalid values
df = df[(df['Close'] > 0) & (df['Volume'] > 0)]

# Sort (IMPORTANT)
df.sort_values(['Ticker', 'Date'], inplace=True)
df.reset_index(drop=True, inplace=True)

# =========================
# HANDLE MISSING VALUES (FIXED)
# Only forward-fill numeric columns → keeps Ticker safe
df[['Close', 'Volume']] = df.groupby('Ticker')[['Close', 'Volume']].ffill()

# =========================
# REMOVE OUTLIERS FUNCTION
def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return data[(data[column] >= lower) & (data[column] <= upper)]

# Apply per stock
df = df.groupby('Ticker', group_keys=False).apply(lambda x: remove_outliers(x, 'Close'))
df = df.groupby('Ticker', group_keys=False).apply(lambda x: remove_outliers(x, 'Volume'))

# FEATURE ENGINEERING
df['Close_Smoothed'] = (
    df.groupby('Ticker')['Close']
    .rolling(window=5)
    .mean()
    .reset_index(level=0, drop=True)
)


# FINAL CLEANUP
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# =========================
# OUTPUT
# =========================
print("Final Shape:", df.shape)
print(df.head())

# Optional save
df.to_csv("cleaned_stock_data.csv", index=False)

/tmp/ipykernel_3283/1381867438.py:41: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('Ticker', group_keys=False).apply(lambda x: remove_outliers(x, 'Close'))
/tmp/ipykernel_3283/1381867438.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('Ticker', group_keys=False).apply(lambda x: remove_outliers(x, 'Volume'))


Final Shape: (13463, 20)
        Date Ticker       Open       High        Low      Close     Volume  \
0 2016-02-29   AAPL  21.957425  22.267995  21.909820  21.918888  140865200   
1 2016-03-01   AAPL  22.136515  22.843794  22.084375  22.789389  201628400   
2 2016-03-02   AAPL  22.784853  22.870995  22.587630  22.839258  132678400   
3 2016-03-03   AAPL  22.800717  23.056879  22.771246  23.009274  147822800   
4 2016-03-04   AAPL  23.206500  23.519335  22.979808  23.351583  184220400   

       SMA_7     SMA_21     EMA_12     EMA_26     RSI_14      MACD  \
0  21.829505  21.699797  21.812767  21.848126  56.325411 -0.035359   
1  21.974913  21.739904  21.963016  21.917849  66.198936  0.045168   
2  22.100242  21.792156  22.097823  21.986101  69.518150  0.111722   
3  22.320781  21.873440  22.238046  22.061892  73.242028  0.176154   
4  22.544558  21.950947  22.409359  22.157425  75.055648  0.251935   

   MACD_Signal  Bollinger_Upper  Bollinger_Lower  Daily_Return  Volatility_7d  \
0   

In [10]:
df_cleaned = pd.read_csv('cleaned_stock_data.csv')
display(df_cleaned.head())

,Date,Ticker,Open,High,Low,Close,Volume,SMA_7,SMA_21,EMA_12,EMA_26,RSI_14,MACD,MACD_Signal,Bollinger_Upper,Bollinger_Lower,Daily_Return,Volatility_7d,Next_Day_Close,Close_Smoothed
0,2016-02-29,AAPL,21.957425,22.267995,21.909820,21.918888,140865200,21.829505,21.699797,21.812767,21.848126,56.325411,-0.035359,-0.095013,22.256425,21.118435,-0.002270,0.012012,22.789389,21.814609
1,2016-03-01,AAPL,22.136515,22.843794,22.084375,22.789389,201628400,21.974913,21.739904,21.963016,21.917849,66.198936,0.045168,-0.066977,22.492944,20.986657,0.039715,0.018828,22.839258,22.079387
2,2016-03-02,AAPL,22.784853,22.870995,22.587630,22.839258,132678400,22.100242,21.792156,22.097823,21.986101,69.518150,0.111722,-0.031237,22.686447,20.946849,0.002188,0.018873,23.009274,22.290210
3,2016-03-03,AAPL,22.800717,23.056879,22.771246,23.009274,147822800,22.320781,21.873440,22.238046,22.061892,73.242028,0.176154,0.010241,22.899146,20.862685,0.007444,0.014179,23.351583,22.505113
4,2016-03-04,AAPL,23.206500,23.519335,22.979808,23.351583,184220400,22.544558,21.950947,22.409359,22.157425,75.055648,0.251935,0.058580,23.165941,20.741200,0.014877,0.014178,23.093153,22.781678
